# Stage 3: learning the convection coefficient $h$

`pinn_thermal.ipynb` trains a PINN to reproduce a temperature field. Here the
goal changes: **pretend we don't know $h$** (how strongly the top surface loses
heat to the air) and let training recover it from the temperature data.

$h$ never appears in the stage 1 loss. The heat equation inside the wall only
contains $\alpha$. $h$ lives in the **top boundary condition**, which stage 1
never enforced. So this notebook adds a third loss term, the boundary condition,
and makes the $h$ inside it trainable.

Cells marked **TODO** are left for you to fill in. Everything else is copied
from `pinn_thermal.ipynb`.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(123)
np.random.seed(123)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## Load the solver output

Same as stage 1, plus `k` (assumed known). **Do not read `d["h"]` here.** The
true value is only loaded in the comparison cell at the end.

In [ ]:
d = np.load("data/baseline.npz")
z, ts, Ts = d["z"], d["t"], d["T"]
H, t_end = float(z[-1]), float(ts[-1])
alpha = float(d["alpha"])
T_inf, T_nozzle, Tg, LH = float(d["T_inf"]), float(d["T_nozzle"]), float(d["Tg"]), float(d["LH"])

print(f"field shape {Ts.shape},  H = {H*1e3:.1f} mm,  t_end = {t_end:.1f} s")
k = float(d["k"])   # conductivity is treated as known; h is not